## IDW

In [ ]:
import pandas as pd
import numpy as np

# Haversine distance (km)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# IDW function
def idw_predict(values, distances, power=2):
    # keep only stations with non-missing rainfall
    mask = ~np.isnan(values)
    values = values[mask]
    distances = distances[mask]
    if len(values) == 0:
        return np.nan  # nothing to interpolate
    weights = 1 / (distances**power)
    weights /= weights.sum()
    return np.sum(weights * values)

# Load rainfall data
rain_df = pd.read_csv("oliwa_MCAR15%.csv")  
rain_df.set_index("Date", inplace=True)

# Load station coordinates
coords = pd.read_csv("stations_coordinates.csv")
coords.set_index("Station", inplace=True)

# Target station: Oliwa
target_station = "oliwa"
target_lat, target_lon = coords.loc[target_station, ["Latitude", "Longitude"]]

# Distances from other stations to target
others = coords.drop(target_station)
distances = others.apply(
    lambda row: haversine(target_lat, target_lon, row["Latitude"], row["Longitude"]),
    axis=1
)

# Fill missing values in Oliwa
filled = []
for idx, row in rain_df.iterrows():
    if pd.isna(row[target_station]):  # if missing
        values = row[others.index].values
        pred = idw_predict(values, distances.values, power=2)
        filled.append(pred)
    else:
        filled.append(row[target_station])

rain_df[f"{target_station}_filled"] = filled

# Save results
rain_df.to_csv("rainfall_filled_IDW15%.csv")

print(rain_df[[target_station, f"{target_station}_filled"]].head(30))

## CCWM

In [ ]:
import pandas as pd
import numpy as np

# Load your dataset
df = pd.read_csv("oliwa_MCAR15%.csv", parse_dates=["Date"])

target_station = "oliwa"
other_stations = [c for c in df.columns if c not in ["Date", target_station]]

# Step 1: Compute correlations with target
correlations = {}
for col in other_stations:
    corr = df[[target_station, col]].corr().iloc[0, 1]  # Pearson correlation
    correlations[col] = corr if not np.isnan(corr) else 0

# Step 2: Convert correlations to weights (absolute value, normalized)
weights = {col: abs(val) for col, val in correlations.items()}
total = sum(weights.values())
weights = {col: val / total for col, val in weights.items()}

print("Normalized Weights (CCWM):")
for col, w in weights.items():
    print(f"{col}: {w:.3f}")

# Step 3: Predict 'oliwa' using weighted sum of other stations
def ccwm_predict(row, weights):
    return sum(row[col] * weights[col] for col in weights)

df[target_station + "_ccwm"] = df[other_stations].apply(lambda row: ccwm_predict(row, weights), axis=1)

# Step 4: Fill missing 'oliwa' with CCWM predictions
df[target_station + "_filled"] = df[target_station].copy()
df.loc[df[target_station].isna(), target_station + "_filled"] = df.loc[df[target_station].isna(), target_station + "_ccwm"]

# Save results
df.to_csv("rainfall_ccwm_filled15%.csv", index=False)

print(df[[target_station, target_station + "_ccwm", target_station + "_filled"]].head(20))

## Random Forest

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# Load dataset with missing values
df = pd.read_csv("oliwa_MCAR15%.csv", parse_dates=["Date"])

# Define target and features manually
target_col = "oliwa"
feature_cols = ["sopot","orgodowa","matemblewo","klukowo","kokoszki","osowa","reja"]  # <-- change to your station names

# Extract X (predictors) and y (target)
X = df[feature_cols]
y = df[target_col]

# Identify missing rows
mask_missing = y.isna()

# Split into training (non-missing) and test (missing)
X_train = X[~mask_missing]
y_train = y[~mask_missing]
X_missing = X[mask_missing]

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=None, n_jobs=None)
rf.fit(X_train, y_train)

# Predict missing values
y_pred = rf.predict(X_missing)

# Fill missing values
df.loc[mask_missing, target_col] = y_pred

# Save imputed dataset
df.to_csv("Oliwa_RF_imputed15%.csv", index=False)

print(f"Imputed {len(y_pred)} missing values in '{target_col}' using {feature_cols}.")

## Multiple Linear Regression 

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import numpy as np

# Load your data (assuming it's in CSV format)
df = pd.read_csv("oliwa_MCAR15%.csv", parse_dates=["Date"])

# Example: predict missing values for 'oliwa'
target_station = "oliwa"

# Define predictors = all stations except target
predictors = [col for col in df.columns if col not in ["Date", target_station]]

# Split rows into "known" (no missing) and "missing" for target
known = df[df[target_station].notnull()]
missing = df[df[target_station].isnull()]

# Training data
X = known[predictors]
y = known[target_station]

# Train MLR model
model_MLR = LinearRegression()
model_MLR.fit(X, y)

# Predict missing values
if not missing.empty:
    X_missing = missing[predictors]
    df.loc[missing.index, target_station] = model_MLR.predict(X_missing)

# Now df[target_station] has filled values
print(df.head())
df.to_csv("Rainfallfilled_MLR15%.csv", index=False)

## XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

# Load your data
df = pd.read_csv("oliwa_MCAR15%.csv", parse_dates=["Date"])

# Target station
target_station = "oliwa"

# Predictors
predictors = [col for col in df.columns if col not in ["Date", target_station]]

# Separate known and missing rows
known = df[df[target_station].notnull()]
missing = df[df[target_station].isnull()]

# Training data
X = known[predictors]
y = known[target_station]

# Train model
modelXGB = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=1.0,
    colsample_bytree=0.8,
    random_state=None
)
modelXGB.fit(X, y)

# Predict missing values
evaluated_values = pd.DataFrame()
if not missing.empty:
    X_missing = missing[predictors]
    predictions = modelXGB.predict(X_missing)

    # Store only evaluated values with their dates
    evaluated_values = pd.DataFrame({
        "Date": missing["Date"],
        f"{target_station}_predicted": predictions
    })

    # Fill them back into the main dataset
    df.loc[missing.index, target_station] = predictions

# Save both
df.to_csv("rainfall_filled15%_xgb.csv", index=False)                     # full dataset
#evaluated_values.to_csv("rainfall_oliwa_predicted_onlyxgb.csv", index=False)  # only evaluated values

print("✅ Full dataset saved to rainfall_filledJULY_xgb.csv")
print("✅ Predicted values only saved to rainfall_oliwa_predicted_only.csv")

## Multilayer Perceptron

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Load your data
df = pd.read_csv("oliwa_MCAR15%.csv", parse_dates=["Date"])

# Choose target station to predict
target_station = "oliwa"

# Use all other stations as predictors
predictors = [col for col in df.columns if col not in ["Date", target_station]]

# Separate rows where target is known vs missing
known = df[df[target_station].notnull()]
missing = df[df[target_station].isnull()]

# Training data
X = known[predictors]
y = known[target_station]

# Define MLP model (with scaling)
model = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(128, 64),   # two hidden layers
                 activation='relu',
                 solver='adam',
                 learning_rate_init=0.001,
                 max_iter=500,
                 random_state=42)
)

# Train the model
model.fit(X, y)

# Predict missing values
if not missing.empty:
    X_missing = missing[predictors]
    df.loc[missing.index, target_station] = model.predict(X_missing)

# Save filled dataset
df.to_csv("rainfall_filled15%_mlp.csv", index=False)

print("✅ Missing values predicted with MLP and saved to rainfall_filled_mlp.csv")

## Mean

In [ ]:
import pandas as pd

# Load data
df = pd.read_csv("rainfall_with_april_missing.csv", parse_dates=["Date"])

# Define target and predictors
target = "oliwa"
predictors = [c for c in df.columns if c not in ["Date", target]]

# Fill NaN in oliwa with row-wise mean of predictors
df[target] = df[target].fillna(df[predictors].mean(axis=1))

# Save result
df.to_csv("rainfall_filled_meanapril.csv", index=False)

print("✅ Missing values in 'oliwa' filled with row-wise mean of other stations.")